# H2: Data Re-uploading Classifier vs Standard Feature Map

**Hypothesis**: A data re-uploading ansatz outperforms a single-pass feature map for the same parameter budget on a toy 4-class dataset.

In [ ]:
import pennylane as qml
import numpy as np
import json
import os
from datetime import datetime

np.random.seed(42)
qml.numpy.random.seed(42)

N_QUBITS = 4
N_CLASSES = 4
N_SAMPLES = 120
N_EPOCHS = 60
LR = 0.05
BATCH_SIZE = 24
N_REUPLOAD = 3

dev = qml.device('lightning.qubit', wires=N_QUBITS, shots=None)

def standard_feature_map(x, params):
    for i in range(N_QUBITS):
        qml.RX(x[i % len(x)], wires=i)
    idx = 0
    for l in range(2):
        for i in range(N_QUBITS):
            qml.RY(params[idx], wires=i); idx += 1
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(N_QUBITS):
            qml.RZ(params[idx], wires=i); idx += 1

def reupload_ansatz(x, params):
    idx = 0
    for r in range(N_REUPLOAD):
        for i in range(N_QUBITS):
            qml.RX(x[i % len(x)], wires=i)
        for i in range(N_QUBITS):
            qml.RY(params[idx], wires=i); idx += 1
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(N_QUBITS):
            qml.RZ(params[idx], wires=i); idx += 1

@qml.qnode(dev)
def standard_circuit(x, params):
    standard_feature_map(x, params)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

@qml.qnode(dev)
def reupload_circuit(x, params):
    reupload_ansatz(x, params)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

In [ ]:
def generate_multiclass_dataset(n_samples, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, size=(n_samples, 2))
    y = np.zeros(n_samples, dtype=int)
    for i, (x1, x2) in enumerate(X):
        if x1 > 0 and x2 > 0:
            y[i] = 0
        elif x1 > 0 and x2 <= 0:
            y[i] = 1
        elif x1 <= 0 and x2 > 0:
            y[i] = 2
        else:
            y[i] = 3
    return X, y

def to_onehot(y, n_classes):
    return np.eye(n_classes)[y]

def train(circuit, X, y, label, n_params):
    y_onehot = to_onehot(y, N_CLASSES)
    params = np.random.default_rng(42).uniform(-0.1, 0.1, size=n_params)
    opt = qml.GradientDescentOptimizer(stepsize=LR)
    losses = []
    for epoch in range(N_EPOCHS):
        idx = np.random.default_rng(epoch).permutation(len(X))
        epoch_loss = 0.0
        for start in range(0, len(X), BATCH_SIZE):
            batch_idx = idx[start:start + BATCH_SIZE]
            batch_X, batch_y = X[batch_idx], y_onehot[batch_idx]
            def loss_fn(p):
                preds = np.array([circuit(x, p) for x in batch_X])
                return np.mean((preds - batch_y) ** 2)
            params, cost = opt.step_and_cost(loss_fn, params)
            epoch_loss += cost
        avg_loss = epoch_loss / max(1, (len(X) // BATCH_SIZE))
        losses.append(float(avg_loss))
        if epoch % 10 == 0:
            preds = np.array([circuit(x, params) for x in X])
            pred_classes = np.argmax(preds, axis=1)
            acc = np.mean(pred_classes == y)
            print(f"[{label}] Epoch {epoch:3d}  loss={avg_loss:.4f}  acc={acc:.3f}")
    final_preds = np.array([circuit(x, params) for x in X])
    pred_classes = np.argmax(final_preds, axis=1)
    final_acc = float(np.mean(pred_classes == y))
    return {'params': params.tolist(), 'losses': losses, 'final_acc': final_acc, 'label': label}

X, y = generate_multiclass_dataset(N_SAMPLES)
os.makedirs('results/H2', exist_ok=True)

n_standard = 2 * (2 * N_QUBITS)
n_reupload = N_REUPLOAD * (2 * N_QUBITS)

print("=== Training standard feature map ===")
standard_result = train(standard_circuit, X, y, 'standard', n_standard)
print("\n=== Training data re-uploading ===")
reupload_result = train(reupload_circuit, X, y, 'reupload', n_reupload)

results = {'standard': standard_result, 'reupload': reupload_result}
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'results/H2/run_{ts}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to results/H2/run_{ts}.json")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(standard_result['losses'], label='Single-pass feature map', marker='o', ms=3)
plt.plot(reupload_result['losses'], label='Data re-uploading', marker='s', ms=3)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('H2: Standard vs Data Re-uploading Training Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
os.makedirs('results/H2', exist_ok=True)
plt.savefig('results/H2/plot.png', dpi=120)
plt.show()
print(f"Final accuracy standard={standard_result['final_acc']:.3f}  reupload={reupload_result['final_acc']:.3f}")